In [15]:
%load_ext autoreload
%autoreload 2

#Pega os dados do seges
from request_seges import Seges as sg
from request_seges import Login
from urllib.parse import urlparse
import urllib3

import requests

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:


session = requests.Session()
usuario = '10631094776'
etapa = 0 # significa que é o trimestre (0-1ºTrimestre, 1-2ºTrimestre...)
senha = usuario


login = Login("https://seges.sedu.es.gov.br")

session_logada, base_url = login.autenticar(usuario, senha)
seges = sg(session_logada, etapa, base_url)

# pega links das turmas
url = 'https://seges.sedu.es.gov.br/avaliacao_modo_avancados/turmas'
lancar_notas = seges.get_minhas_turmas(url)

# pega minhas notas
listagem_avaliacao = seges.get_links_minhas_notas(lancar_notas['href'])
listagem_avaliacao.tail()
'''
output
classroom é a turma
discipline_id é o tipo de diciplina
stage_id é o código do trimestre
'''

minhas_turmas = (listagem_avaliacao.drop_duplicates(subset='classroom_id').reset_index(drop=True)) # só serve para pegar o nome dos alunos e fim
meus_alunos = seges.get_alunos_por_turma(listagem_avaliacao['href'].to_list())
minhas_avaliacoes = seges.get_avaliacoes(listagem_avaliacao['href'])
listagem_avaliacao["classroom_id"] = listagem_avaliacao["classroom_id"].astype(int)

notas = seges.get_notas(listagem_avaliacao['href'])

In [17]:
df = notas.copy()

df = df.merge(
    meus_alunos,
    on=["turma", "classroom_id", "aluno_id", "classroom_evaluation_id"],
    how="left"
)

df = df.merge(
    minhas_avaliacoes,
    on=["turma", "classroom_evaluation_id"],
    how="left"
)

df["discipline_id"] = df["discipline_id"].astype(int)
listagem_avaliacao["discipline_id"] = listagem_avaliacao["discipline_id"].astype(int)

df = df.merge(
    listagem_avaliacao,
    on=["turma", "classroom_id", "discipline_id"],
    how="left"
)


In [ ]:

df['result']= df[['number', 'recovery']].max(axis=1)

,turma,classroom_id,aluno_id,classroom_evaluation_id,number,recovery,discipline_id,numero,nome,condicao,avaliacao_nome,atividade_nome,etapa,disciplina,stage_id,href,result
0,2ªV01-EM-LCH,39516,765155,1730132,5.0,None,261703,1,ACSA CRISTINA VITORIA BERNARDO KROHLING,active,LISTAS DE EXERCICIOS,LISTAS DE EXERCICIOS,0,QUÍMICA,5222,https://seges.sedu.es.gov.br/grades?classroom_...,5.0
1,2ªV01-EM-LCH,39516,765536,1730132,5.0,None,261703,29,MIRELA FERREIRA DE JESUS,active,LISTAS DE EXERCICIOS,LISTAS DE EXERCICIOS,0,QUÍMICA,5222,https://seges.sedu.es.gov.br/grades?classroom_...,5.0
2,2ªV01-EM-LCH,39516,765537,1730132,5.0,None,261703,31,NICOLAS BATISTA DE OLIVEIRA,active,LISTAS DE EXERCICIOS,LISTAS DE EXERCICIOS,0,QUÍMICA,5222,https://seges.sedu.es.gov.br/grades?classroom_...,5.0
3,2ªV01-EM-LCH,39516,765541,1730132,NaN,None,261703,33,RAPHAELA VITORIA NOGUEIRA DE JESUS,active,LISTAS DE EXERCICIOS,LISTAS DE EXERCICIOS,0,QUÍMICA,5222,https://seges.sedu.es.gov.br/grades?classroom_...,None
4,2ªV01-EM-LCH,39516,765550,1730132,4.0,None,261703,19,LAURIENY RODRIGUES ARAUJO,active,LISTAS DE EXERCICIOS,LISTAS DE EXERCICIOS,0,QUÍMICA,5222,https://seges.sedu.es.gov.br/grades?classroom_...,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,3ªV01-EM-MOD,35914,895844,1785290,NaN,None,257281,30,MATHEUS ALOCHIO VERONESI DE VARGAS,active,PROVA,PROVA,0,QUÍMICA,5222,https://seges.sedu.es.gov.br/grades?classroom_...,None
208,3ªV01-EM-MOD,35914,895846,1785290,8.0,None,257281,31,MATHEUS POSSE SANTOS DE ABREU,active,PROVA,PROVA,0,QUÍMICA,5222,https://seges.sedu.es.gov.br/grades?classroom_...,8.0
209,3ªV01-EM-MOD,35914,895848,1785290,8.0,None,257281,32,MAYSA CHRISTINA MARQUES SOUZA,active,PROVA,PROVA,0,QUÍMICA,5222,https://seges.sedu.es.gov.br/grades?classroom_...,8.0
210,3ªV01-EM-MOD,35914,898849,1785290,4.0,None,257281,21,KAUA FERNANDES DOS ANJOS,active,PROVA,PROVA,0,QUÍMICA,5222,https://seges.sedu.es.gov.br/grades?classroom_...,4.0


In [19]:
import os
import pandas as pd
with pd.ExcelWriter("output/diario.xlsx", engine="xlsxwriter") as writer:

    for turma in df["turma"].dropna().unique():

        df_turma = df[df["turma"] == turma].copy()

        # garante nomes limpos de sheet (limite Excel = 31 chars)
        sheet_name = str(turma)[:31]

        # =========================
        # pivot: aluno x avaliação
        # =========================
        df_turma["atividade_coluna"] = (
            df_turma["avaliacao_nome"].astype(str) +
            " - " +
            df_turma["disciplina"].astype(str)
        )

        df_pivot = df_turma.pivot_table(
            index=["numero", "nome"],
            columns="atividade_coluna",
            values="result",
            aggfunc="max"
        ).reset_index()

        # remove colunas vazias
        df_pivot = df_pivot.dropna(axis=1, how="all")

        # ordena colunas
        fixas = ["numero", "nome"]
        cols = [c for c in df_pivot.columns if c not in fixas]

        df_pivot = df_pivot[fixas + sorted(cols)]

        # exporta
        df_pivot.to_excel(writer, sheet_name=sheet_name, index=False)

        # ajuste automático de largura
        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes(1, 2)

        for i, col in enumerate(df_pivot.columns):
            max_len = max(df_pivot[col].fillna("").astype(str).str.len().max(), len(str(col))
)
            worksheet.set_column(i, i, max_len + 2)